In [1]:
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
from sklearn.metrics import confusion_matrix
from lightgbm.callback import early_stopping


In [2]:
TRAIN_LOCS_KEY = 'train_locs'
TRAIN_IDS_KEY = 'train_ids'
TAXON_IDS_KEY = 'taxon_ids'
TAXON_NAME_KEY = 'taxon_names'

TEST_LOCS_KEY = 'test_locs'
TEST_IDS_KEY = 'test_pos_inds'

Reading the file:

In [3]:
filepath = os.path.join(os.getcwd(), 'species_train.npz')
data = np.load(filepath, allow_pickle=True)
train_locs = data[TRAIN_LOCS_KEY]
train_ids = data[TRAIN_IDS_KEY]
taxon_ids = data[TAXON_IDS_KEY]
taxon_names = data[TAXON_NAME_KEY]

Reading the test file:

In [4]:
test_filepath = os.path.join(os.getcwd(), 'species_test.npz')
test_data = np.load(test_filepath, allow_pickle=True)
test_locs = test_data[TEST_LOCS_KEY]
test_ids = test_data[TEST_IDS_KEY]

Mapping the taxon ids to taxon latin names: 

In [5]:
species_ids_names = dict(zip(data['taxon_ids'], data['taxon_names']))  # latin names of species 

Create pandas Dataframe from the train data: 

In [6]:
df = pd.DataFrame({
    'latitude': train_locs[:, 0],
    'longitude': train_locs[:, 1], 
    'taxon_id': data[TRAIN_IDS_KEY]
})
df['taxon_name'] = [species_ids_names[id] for id in data[TRAIN_IDS_KEY].astype(int)]
df.head()

,latitude,longitude,taxon_id,taxon_name
0,-18.286728,143.481247,31529,Lophognathus gilberti
1,-13.099798,130.783646,31529,Lophognathus gilberti
2,-13.965274,131.695145,31529,Lophognathus gilberti
3,-12.853950,132.800507,31529,Lophognathus gilberti
4,-12.196790,134.279327,31529,Lophognathus gilberti


Create pandas Dataframe from the test data: 

In [7]:
rows = [
    [test_locs[loc_id][0], test_locs[loc_id][1], taxon_id]
    for taxon_id, loc_ids in zip(taxon_ids, test_ids)
    for loc_id in loc_ids
]

In [8]:
test_df = pd.DataFrame(rows, columns=["latitude", "longitude", "taxon_id"])
test_df['taxon_name'] = [species_ids_names[id] for id in test_df["taxon_id"].astype(int)]
test_df.head()

,latitude,longitude,taxon_id,taxon_name
0,-19.884237,126.052979,31529,Lophognathus gilberti
1,-20.219316,124.723953,31529,Lophognathus gilberti
2,-20.053690,125.386505,31529,Lophognathus gilberti
3,-19.973000,126.462440,31529,Lophognathus gilberti
4,-19.962839,124.980362,31529,Lophognathus gilberti


In [9]:
test_df.shape

(1706646, 4)

Data Cleanining: 

<small>1. Check for missing or invalid coordinates:</small>

In [10]:
df = df.dropna(subset=['latitude', 'longitude'])
test_df = test_df.dropna(subset=['latitude', 'longitude'])
df = df[(df['latitude'].between(-90, 90)) & (df['longitude'].between(-180, 180))]
test_df = test_df[(test_df['latitude'].between(-90, 90)) & (test_df['longitude'].between(-180, 180))]
df.shape, test_df.shape

((272037, 4), (1706646, 4))

<small>2. Remove any duplicates or nearly duplicates (observations that are extremely close):</small>

In [11]:
df['lat_rounded'] = df['latitude'].round(5)
df['lon_rounded'] = df['longitude'].round(5)
test_df['lat_rounded'] = test_df['latitude'].round(5)
test_df['lon_rounded'] = test_df['longitude'].round(5)

In [12]:
df = df.drop_duplicates(subset=['lat_rounded', 'lon_rounded'])
test_df = test_df.drop_duplicates(subset=['lat_rounded', 'lon_rounded'])
df.shape, test_df.shape

((237977, 6), (253243, 6))

<small>4. Validate species IDs: </small>

In [13]:
df['taxon_id'].isna().sum(), test_df['taxon_id'].isna().sum()

(np.int64(0), np.int64(0))

<small>5. Only keep birds:</small>

<small>Note: Only run the next 2 blocks one time as they take a few seconds:</small>

In [14]:
taxa = pd.read_csv('taxa.csv')
birds = taxa[taxa['class'] == 'Aves']
bird_taxon_ids = set(birds['id'])
len(bird_taxon_ids)

35102

In [15]:
df = df[df['taxon_id'].isin(bird_taxon_ids)].copy()
test_df = test_df[test_df['taxon_id'].isin(bird_taxon_ids)].copy()
df.shape, test_df.shape

((151391, 6), (230110, 6))

<small>6. Convert to categorical labels:</small>

In [16]:
le = LabelEncoder()
le.fit(pd.concat([df['taxon_id'], test_df['taxon_id']]))
df['label'] = le.transform(df['taxon_id'])
test_df['label'] = le.transform(test_df['taxon_id'])

<small>7. Append the climate data</small>

In [17]:
import rasterio
import numpy as np
import os

base_dir = os.getcwd()  # assumes folders are in the same directory as this notebook

def load_stack(folder_name):
    folder = os.path.join(base_dir, folder_name)
    files = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".tif")])
    if len(files) != 12:
        raise RuntimeError(f"Expected 12 GeoTIFFs in {folder_name}, found {len(files)}.")
    layers = []
    transform = None
    for f in files:
        with rasterio.open(f) as src:
            layers.append(src.read(1))
            transform = src.transform  # all months share the same transform
    return np.stack(layers), transform  # shape: (12, H, W), and affine transform

tmin, transform = load_stack("wc2.1_2.5m_tmin")
tmax, _         = load_stack("wc2.1_2.5m_tmax")
prec, _         = load_stack("wc2.1_2.5m_prec")

print("Stacks loaded. Shapes:",
      "tmin", tmin.shape, "tmax", tmax.shape, "prec", prec.shape)
print("Transform:", transform)

Stacks loaded. Shapes: tmin (12, 4320, 8640) tmax (12, 4320, 8640) prec (12, 4320, 8640)
Transform: | 0.04, 0.00,-180.00|
| 0.00,-0.04, 90.00|
| 0.00, 0.00, 1.00|


In [18]:
#Cleaning the data again as the precipitation values are very large, so I'll normalize them
from rasterio.transform import rowcol
import numpy as np

def get_climate_for_points(df, transform, tmin, tmax, prec, lat_col="latitude", lon_col="longitude"):
    # Convert lat/lon → raster indices
    rows, cols = rowcol(transform, df[lon_col].values, df[lat_col].values)
    rows = np.clip(rows, 0, tmin.shape[1]-1)
    cols = np.clip(cols, 0, tmin.shape[2]-1)

    # Helper: clean up weird values (NoData etc.)
    def clean(arr):
        arr = arr.astype(float)
        arr[arr > 1e4] = np.nan  # remove unrealistic large values
        return arr

    tmin = clean(tmin)
    tmax = clean(tmax)
    prec = clean(prec)

    # Extract values and average across 12 months safely
    tmin_mean = np.nanmean(tmin[:, rows, cols], axis=0)
    tmax_mean = np.nanmean(tmax[:, rows, cols], axis=0)
    prec_mean = np.nanmean(prec[:, rows, cols], axis=0)

    return tmin_mean, tmax_mean, prec_mean

In [19]:
tmin_avg, tmax_avg, prec_avg = get_climate_for_points(df, transform, tmin, tmax, prec)

In [20]:
test_tmin_avg, test_tmax_avg, test_prec_avg = get_climate_for_points(test_df, transform, tmin, tmax, prec)

In [21]:
df["Tmin_avg"] = tmin_avg
df["Tmax_avg"] = tmax_avg
df["Prec_avg"] = prec_avg

df[["latitude","longitude","Tmin_avg","Tmax_avg","Prec_avg"]].head()

,latitude,longitude,Tmin_avg,Tmax_avg,Prec_avg
53,21.086105,-86.852867,20.967000,31.436000,103.416667
54,19.186003,-96.199600,19.327333,31.831333,117.833333
55,17.538877,-89.113724,19.496333,30.576333,125.583333
56,20.648556,-105.220955,19.674123,31.835527,87.750000
57,18.409698,-95.096657,20.515000,29.086000,156.750000


In [22]:
test_df["Tmin_avg"] = test_tmin_avg
test_df["Tmax_avg"] = test_tmax_avg
test_df["Prec_avg"] = test_prec_avg

test_df[["latitude","longitude","Tmin_avg","Tmax_avg","Prec_avg"]].head()

,latitude,longitude,Tmin_avg,Tmax_avg,Prec_avg
784,19.289352,-89.842888,1.885033e+01,3.272100e+01,91.750000
785,20.777590,-88.534241,1.882467e+01,3.272433e+01,103.833333
786,21.513868,-86.804565,2.165833e+01,3.071667e+01,100.250000
787,18.710384,-94.898964,-3.400000e+38,-3.400000e+38,-32768.000000
788,21.509193,-88.522995,1.973447e+01,3.149773e+01,73.250000


In [23]:
out_path = "bird_species_rerun_with_averaged_climate.csv"
df.to_csv(out_path, index=False)
out_path, df.shape

('bird_species_rerun_with_averaged_climate.csv', (151391, 10))

In [24]:
out_path = "test_bird_species_rerun_with_averaged_climate.csv"
test_df.to_csv(out_path, index=False)
out_path, test_df.shape

('test_bird_species_rerun_with_averaged_climate.csv', (230110, 10))

In [25]:
df = pd.read_csv('bird_species_rerun_with_averaged_climate.csv')

In [26]:
test_df = pd.read_csv('test_bird_species_rerun_with_averaged_climate.csv')

<small>8. Clean the climate data</small>

In [27]:
df['Tmin_avg'] = df['Tmin_avg'].mask(df['Tmin_avg'] < -1e+30, np.nan)
df['Tmax_avg'] = df['Tmax_avg'].mask(df['Tmax_avg'] < -1e+30, np.nan)
df['Prec_avg'] = df['Prec_avg'].mask(df['Prec_avg'] < 0, np.nan)
print(f"Shape with nan data: {df.shape}")
df = df.dropna(subset=['Tmin_avg', 'Tmax_avg', 'Prec_avg'])
print(f"Shape without nan data: {df.shape}")

Shape with nan data: (151391, 10)
Shape without nan data: (150194, 10)


In [28]:
len(test_df)

230110

In [29]:
test_df['Tmin_avg'] = test_df['Tmin_avg'].mask(test_df['Tmin_avg'] < -1e+30, np.nan)
test_df['Tmax_avg'] = test_df['Tmax_avg'].mask(test_df['Tmax_avg'] < -1e+30, np.nan)
test_df['Prec_avg'] = test_df['Prec_avg'].mask(test_df['Prec_avg'] < 0, np.nan)
print(f"Shape with nan data: {test_df.shape}")
test_df = test_df.dropna(subset=['Tmin_avg', 'Tmax_avg', 'Prec_avg'])
print(f"Shape without nan data: {test_df.shape}")

Shape with nan data: (230110, 10)
Shape without nan data: (53055, 10)


<small>Note: mention in the report that we lost a lot of data because of removing NaN temperature, in the ociens mainly.</small>

In [30]:
(df['Tmax_avg'] < df['Tmin_avg']).sum(), (test_df['Tmax_avg'] < test_df['Tmin_avg']).sum()

(np.int64(0), np.int64(0))

<small>Add sin feature to capture circularity</small>

In [31]:
df['lat_sin'] = np.sin(np.deg2rad(df['latitude']))
df['lat_cos'] = np.cos(np.deg2rad(df['latitude']))
df['lon_sin'] = np.sin(np.deg2rad(df['longitude']))
df['lon_cos'] = np.cos(np.deg2rad(df['longitude']))

In [32]:
test_df['lat_sin'] = np.sin(np.deg2rad(test_df['latitude']))
test_df['lat_cos'] = np.cos(np.deg2rad(test_df['latitude']))
test_df['lon_sin'] = np.sin(np.deg2rad(test_df['longitude']))
test_df['lon_cos'] = np.cos(np.deg2rad(test_df['longitude']))

In [33]:
df.shape, test_df.shape

((150194, 14), (53055, 14))

<small>9. Split the data to x and y and normalize the climate features</small>

In [34]:
X_data = df.drop(columns=['taxon_id', 'taxon_name', 'lat_rounded', 'lon_rounded', 'label'])
y_data = df['label']
climate_features = ['Tmin_avg', 'Tmax_avg', 'Prec_avg']
non_scaled_features = ['latitude', 'longitude']
# scale climate 
scaler = StandardScaler()
scaler.fit(X_data[climate_features])
X_scaled = X_data.copy()
X_scaled[climate_features] = scaler.transform(X_data[climate_features])
X_scaled.describe()

,latitude,longitude,Tmin_avg,Tmax_avg,Prec_avg,lat_sin,lat_cos,lon_sin,lon_cos
count,150194.000000,150194.000000,1.501940e+05,1.501940e+05,1.501940e+05,150194.000000,150194.000000,150194.000000,150194.000000
mean,15.544443,-9.343506,-1.998304e-16,-6.274977e-16,2.195107e-16,0.242026,0.815972,-0.217722,0.036314
std,32.019650,95.974393,1.000003e+00,1.000003e+00,1.000003e+00,0.510521,0.122410,0.765637,0.604223
min,-75.284950,-178.060320,-6.684864e+00,-6.659755e+00,-1.640339e+00,-0.967201,0.254012,-1.000000,-0.999791
25%,-19.905400,-96.408343,-6.481463e-01,-7.715034e-01,-6.507734e-01,-0.340468,0.741686,-0.968809,-0.465755
50%,27.829345,-47.548143,1.810142e-02,1.161172e-01,-1.759906e-01,0.466840,0.828903,-0.709194,0.047012
75%,41.539592,73.949602,6.747353e-01,7.379541e-01,3.944445e-01,0.663137,0.907944,0.518015,0.540488
max,72.515430,178.827590,2.290127e+00,2.249082e+00,1.336141e+01,0.953798,1.000000,1.000000,1.000000


In [35]:
X_test = test_df.drop(columns=['taxon_id', 'taxon_name', 'lat_rounded', 'lon_rounded', 'label'])
y_test = test_df['label']
climate_features = ['Tmin_avg', 'Tmax_avg', 'Prec_avg']
non_scaled_features = ['latitude', 'longitude']
# scale climate 
X_test_scaled = X_test.copy()
X_test_scaled[climate_features] = scaler.transform(X_test[climate_features])
X_test_scaled.describe()

,latitude,longitude,Tmin_avg,Tmax_avg,Prec_avg,lat_sin,lat_cos,lon_sin,lon_cos
count,53055.000000,53055.000000,53055.000000,53055.000000,53055.000000,53055.000000,53055.000000,53055.000000,53055.000000
mean,24.220822,27.186278,-0.429962,-0.352688,-0.367111,0.359539,0.768096,0.241492,0.312389
std,32.618285,76.175580,1.855896,1.925159,1.220691,0.484355,0.214861,0.660817,0.638303
min,-85.600830,-179.933670,-6.914783,-7.159502,-1.640339,-0.997054,0.076705,-1.000000,-1.000000
25%,0.949113,-6.881928,-1.958520,-1.977172,-1.184687,0.016564,0.600040,-0.115487,-0.222389
50%,27.473444,28.963263,0.140522,0.285981,-0.739469,0.461337,0.852191,0.453236,0.534939
75%,52.553610,84.507910,1.233963,1.323563,0.098792,0.793923,0.953905,0.762581,0.898801
max,82.845260,179.994780,2.635286,2.399846,9.357927,0.992213,1.000000,1.000000,1.000000


<small>9. Split to train and validation sets</small>

In [36]:
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_data,
                                                  test_size=0.2,
                                                  random_state=42, 
                                                  stratify=y_data)

X_train.to_csv('X_train.csv', index=False)
X_val.to_csv('X_val.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_val.to_csv('y_val.csv', index=False)

Exploratory Data Analysis:

In [ ]:
# Hajer

Models:

<small>Model 1: LightGBM<br> This model depends on decision tree and has hyperparameters</small>


In [37]:
# Model 1 (a): LightGBM with probability output
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test_scaled, label=y_test)
boosting_types = ['gbdt']
num_leaves = [31]
max_depths = [-1]
for boosting_type in boosting_types: 
    for num_leaf in num_leaves: 
        for max_depth in max_depths: 
            params = {
                'objective': 'multiclass',
                'num_class': 285,
                'learning_rate': 0.05,
                'metric': 'multi_logloss',
                'boosting_type': "gbdt",
                'max_depth': 30, 
                'num_leaves': 31, 
                'lambda_l1': 0.5,
                'lambda_l2': 1.0, 
                'is_unbalance': True
            }
            clf = lgb.train(
                params,
                train_data,
                valid_sets=[test_data],
                num_boost_round=2000,
                callbacks= [early_stopping(stopping_rounds=100)]
            )
            y_pred_prob = clf.predict(X_test_scaled, num_iteration=clf.best_iteration)
            y_pred = y_pred_prob.argmax(axis=1)
            print("Accuracy:", accuracy_score(y_test, y_pred))
            correct = y_pred == y_test

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2295
[LightGBM] [Info] Number of data points in the train set: 120155, number of used features: 9
[LightGBM] [Info] Start training from score -6.747778
[LightGBM] [Info] Start training from score -6.199370
[LightGBM] [Info] Start training from score -5.010677
[LightGBM] [Info] Start training from score -7.745294
[LightGBM] [Info] Start training from score -8.007658
[LightGBM] [Info] Start training from score -7.846390
[LightGBM] [Info] Start training from score -4.401481
[LightGBM] [Info] Start training from score -5.656283
[LightGBM] [Info] Start training from score -7.745294
[LightGBM] [Info] Start training from score -7.982966
[LightGBM] [Info] Start training from score -6.151360
[LightGBM] [Info] Start training from score -6.359000

In [ ]:
# Model 1 (b): LightGBM on locations only!

In [ ]:
# Model 1 (c): LightGBM with vector binary output

<small>Evaluating LightGBM</small>

In [ ]:
le.classes_.shape

In [38]:
print(f1_score(y_test, y_pred, average='macro'))
print("Top-5 acc", top_k_accuracy_score(y_test, y_pred_prob, k=5, labels=list(set(y_train))))
print("Top-7 acc", top_k_accuracy_score(y_test, y_pred_prob, k=7, labels=list(set(y_train))))
print("Top-10 acc", top_k_accuracy_score(y_test, y_pred_prob, k=10, labels=list(set(y_train))))

0.0263371667775216
Top-5 acc 0.43077937988879467
Top-7 acc 0.5797380077278296
Top-10 acc 0.7393271133729149


In [ ]:
# Model 2:

In [ ]:
# Model 3: